In [16]:
## RAGAS is a framework for evaluating the RAG pipeline.

## Metrics:
# 1. Context Relevancy: Measures how relevant the retrieved context is to the question asked.
#    It filters out irrelevant information from the retrieved context.
#    Calculated as: number of relevant sentences in context / total sentences in context.

# 2. Context Precision: Measures the signal-to-noise ratio of the retrieved context.
#    It evaluates whether the relevant chunks are ranked higher than irrelevant ones.
#    Calculated as: mean of precision@k for each relevant chunk in the ranked retrieved context,
#    where precision@k = number of relevant chunks in top-k / k.

# 3. Context Recall: Measures how much of the ground truth is captured in the retrieved context.
#    It checks if all the necessary information to answer the question was retrieved.
#    Calculated as: Recall@K = number of ground truth sentences attributable to context / total sentences in ground tru th.

# 4. Faithfulness: Measures how factually consistent the generated answer is with the retrieved context.
#    It ensures the answer does not contain information not present in the context (no hallucinations).
#    Calculated as: number of answer statements that can be inferred from context / total statements in answer.

# 5. Answer Relevancy: Measures how relevant the generated answer is to the original question.
#    It penalizes answers that are incomplete or contain redundant information.
#    Calculated as: mean cosine similarity between the original question and n questions
#    generated from the answer, using embeddings to capture semantic similarity.

In [3]:
import nest_asyncio
nest_asyncio.apply()

import ragas
print(ragas.__version__)

import os
import pandas as pd
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))


0.4.3


True

In [6]:
api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

In [10]:
loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


In [11]:
# RAGAS expects a file_name dict as key
for document in docs:
    document.metadata["file_name"] = document.metadata["source"]

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.graph import NodeType
from ragas.testset.transforms.extractors import EmbeddingExtractor, SummaryExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.relationship_builders import CosineSimilarityBuilder, OverlapScoreBuilder
from ragas.testset.transforms.engine import Parallel
from ragas.utils import num_tokens_from_string

# Use OpenAI for testset generation — Euriai returns 403 on some RAGAS internal calls
generator_llm = LangchainLLMWrapper(model)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

query_distribution = default_query_distribution(generator_llm)

# Custom transforms: skip HeadlinesExtractor/HeadlineSplitter which crash when the
# LLM response can't be parsed into headlines.
def filter_doc(node):
    return node.type == NodeType.DOCUMENT and num_tokens_from_string(node.properties["page_content"]) > 100

def filter_docs(node):
    return node.type == NodeType.DOCUMENT

summary_extractor = SummaryExtractor(llm=generator_llm, filter_nodes=filter_doc)
summary_emb_extractor = EmbeddingExtractor(
    embedding_model=generator_embeddings,
    property_name="summary_embedding",
    embed_property_name="summary",
    filter_nodes=filter_doc,
)
cosine_sim_builder = CosineSimilarityBuilder(
    property_name="summary_embedding",
    new_property_name="summary_similarity",
    threshold=0.5,
    filter_nodes=filter_doc,
)
ner_extractor = NERExtractor(llm=generator_llm)
ner_overlap_sim = OverlapScoreBuilder(threshold=0.01)
theme_extractor = ThemesExtractor(llm=generator_llm, filter_nodes=filter_docs)
node_filter = CustomNodeFilter(llm=generator_llm)

custom_transforms = [
    summary_extractor,
    node_filter,
    Parallel(summary_emb_extractor, theme_extractor, ner_extractor),
    Parallel(cosine_sim_builder, ner_overlap_sim),
]

testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=8,
    query_distribution=query_distribution,
    transforms=custom_transforms,
    raise_exceptions=False,
)


C:\Users\C90008809\AppData\Local\Temp\ipykernel_17872\1487550179.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(model)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_17872\1487550179.py:15: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
Applying CustomNodeFilter:   0%|          | 0/3 [00:00<?, ?it/s]c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\euriai\langchain.py:323: RuntimeWarning: c

In [13]:
testset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the key ingredients and characteristi...,"[margherita pizza; $12; classic with tomato, m...",Lasagna is a layered pasta dish that features ...,Culinary Enthusiast,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,What role does Nero d’Avola play in the dining...,"[In the heart of the old quarter of Palermo, a...","At Chef Amico, Nero d’Avola is more than just ...",Restaurant Owner,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What makes cannoli a special dessert in Chef A...,"[In the charming streets of Palermo, tucked aw...",Cannoli is considered the crown jewel of Sicil...,Restaurant Owner,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
3,What role does Margherita pizza play in Chef A...,[<1-hop>\n\nIn the heart of the old quarter of...,Margherita pizza is a classic dish that embodi...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,How does Chef Amico's culinary journey reflect...,[<1-hop>\n\nIn the heart of the old quarter of...,Chef Amico's culinary journey is deeply rooted...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How does Chef Amico's dedication to Sicilian c...,[<1-hop>\n\nIn the heart of the old quarter of...,Chef Amico's dedication to Sicilian cuisine is...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,What experiences in Italy shaped Chef Amico's ...,[<1-hop>\n\nIn the charming streets of Palermo...,Chef Amico's culinary journey was profoundly s...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,What inspired Chef Amico to create his restaur...,[<1-hop>\n\nIn the charming streets of Palermo...,Chef Amico was inspired to create his restaura...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,How did Chef Amico's upbringing in Palermo and...,[<1-hop>\n\nIn the charming streets of Palermo...,"Chef Amico's upbringing in Palermo, where he w...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [14]:
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [15]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])


In [16]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)


In [17]:
df = pd.read_csv("./questions_answers/qa.csv", delimiter=";")
questions = df["question"].tolist()
ground_truth = df["ground_truth"].tolist()


In [18]:
df.head()

,question,ground_truth
0,Where was Amico born?,Amico was born in the heart of the old quarter...
1,What was Amico's early culinary influence?,Amico was influenced by the cooking in his Non...
2,What skill did Amico learn from Palermo's mark...,Amico learned to select the freshest fish and ...
3,Where in Italy did Amico gain culinary experie...,Amico gained culinary experience across variou...
4,"What is ""Chef Amico"" restaurant known for?",Chef Amico is known for combining Sicilian and...


In [19]:
# RAGAS 0.4.x uses EvaluationDataset with SingleTurnSample
# Fields: user_input (question), response (answer), retrieved_contexts, reference (ground_truth)
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

samples = []
for query, gt in zip(questions, ground_truth):
    answer = rag_chain.invoke(query)
    contexts = [doc.page_content for doc in retriever.invoke(query)]
    samples.append(
        SingleTurnSample(
            user_input=query,
            response=answer,
            retrieved_contexts=contexts,
            reference=gt,
        )
    )

dataset = EvaluationDataset(samples=samples)


In [21]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    ContextRelevance,
)

eval_llm = LangchainLLMWrapper(model)
eval_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    ContextRelevance(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
    Faithfulness(),
    AnswerRelevancy(),
]


C:\Users\C90008809\AppData\Local\Temp\ipykernel_17872\506273035.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_17872\506273035.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_17872\506273035.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipy

In [22]:
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings
)

print(result)

Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]Exception raised in Job[0]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'ContextRelevance._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[1]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'LLMContextPrecisionWithReference._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[2]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'LLMContextRecall._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[3]: RuntimeError(Timeout should be used inside a ta

Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]Exception raised in Job[0]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'ContextRelevance._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[1]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'LLMContextPrecisionWithReference._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[2]: RuntimeError(Timeout should be used inside a task)
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\ragas\executor.py:84: RuntimeWarning: coroutine 'LLMContextRecall._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[3]: RuntimeError(Timeout should be used inside a ta

{'nv_context_relevance': nan, 'llm_context_precision_with_reference': nan, 'context_recall': nan, 'faithfulness': nan, 'answer_relevancy': nan}


In [23]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,nv_context_relevance,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,Where was Amico born?,[Amico's life was deeply entwined with the vib...,Amico was born in the heart of the old quarter...,Amico was born in the heart of the old quarter...,NaN,NaN,NaN,NaN,NaN
1,What was Amico's early culinary influence?,[Amico's life was deeply entwined with the vib...,Amico's early culinary influence was his Nonna...,Amico was influenced by the cooking in his Non...,NaN,NaN,NaN,NaN,NaN
2,What skill did Amico learn from Palermo's mark...,"[From a young age, Amico was immersed in the a...",Amico learned to choose the freshest fish from...,Amico learned to select the freshest fish and ...,NaN,NaN,NaN,NaN,NaN
3,Where in Italy did Amico gain culinary experie...,"[As he grew, so did his desire to explore beyo...",Amico gained culinary experience in various re...,Amico gained culinary experience across variou...,NaN,NaN,NaN,NaN,NaN
4,"What is ""Chef Amico"" restaurant known for?",[Chef Amico’s doors opened to a world where th...,Chef Amico restaurant is known for its warm an...,Chef Amico is known for combining Sicilian and...,NaN,NaN,NaN,NaN,NaN
5,What does Amico's restaurant menu reflect?,"[At Chef Amico, every dish told a story. The m...",Amico's restaurant menu reflects a tapestry of...,The menu reflects Amico's culinary journey and...,NaN,NaN,NaN,NaN,NaN
6,How does Amico perceive hospitality?,"[For Amico, hospitality was an art form. He be...",Amico perceives hospitality as an art form and...,Amico sees hospitality as an art of celebratin...,NaN,NaN,NaN,NaN,NaN
7,"What distinguishes ""Chef Amico"" in Palermo?","[Returning to Palermo with a vision, Amico ope...","""Chef Amico"" is distinguished in Palermo by it...",Chef Amico is distinguished by Amico's dedicat...,NaN,NaN,NaN,NaN,NaN
8,What activities does Amico engage in besides c...,[Amico's life was deeply entwined with the vib...,"Besides cooking, Amico engages in mentoring yo...","Amico mentors young chefs, conducts culinary w...",NaN,NaN,NaN,NaN,NaN
9,How is Amico's legacy beyond his dishes?,[Amico’s legacy is not just in the dishes he c...,Amico's legacy extends beyond his dishes throu...,Amico's legacy lies in his community involveme...,NaN,NaN,NaN,NaN,NaN


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = result.to_pandas()

heatmap_data = df[
    [
        "context_relevance",
        "llm_context_precision_with_reference",
        "llm_context_recall",
        "faithfulness",
        "answer_relevancy",
    ]
]

cmap = LinearSegmentedColormap.from_list("green_red", ["red", "green"])

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", linewidths=0.5, cmap=cmap)

# RAGAS 0.4.x uses 'user_input' instead of 'question'
plt.yticks(ticks=range(len(df["user_input"])), labels=df["user_input"], rotation=0)

plt.show()
